# Social Media Engagement Analysis: Initial Data Inspection & Quality Audit
**College Data Visualization (DV) Project**  
**Task Phase**: Milestone 1 - Dataset Inspection & Integrity Audit  
**Author**: Data Analytics & Visualization Team  
**Objective**: Systematically inspect raw social media dataset, verify schema against DV project requirements, uncover anomalies and quality defects without modifying raw data, and formulate cleaning treatments for stakeholder approval.


## 1. Environment Setup and Raw Data Loading
We load the raw dataset located in the `data/` folder using relative paths to ensure cross-platform reproducibility.
The dataset is loaded into a pandas DataFrame without mutating the underlying CSV file.


In [1]:
import os
import pandas as pd
import numpy as np

# Cross-platform relative path resolution (supports execution from project root or notebooks/ directory)
csv_path = os.path.join('data', 'social_media_sentiment_raw.csv') if os.path.exists(os.path.join('data', 'social_media_sentiment_raw.csv')) else os.path.join('..', 'data', 'social_media_sentiment_raw.csv')

print(f"Loading raw dataset from: {csv_path}")
print(f"File exists: {os.path.exists(csv_path)}")
print(f"File size: {os.path.getsize(csv_path):,} bytes")

# Load raw dataset without modifying original file
df_raw = pd.read_csv(csv_path)
print(f"Dataset successfully loaded with shape: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns.")


Loading raw dataset from: data\social_media_sentiment_raw.csv
File exists: True
File size: 346,486 bytes
Dataset successfully loaded with shape: 2,260 rows and 13 columns.


## 2. Structural Inspection: Dimensions, Schema, and Sample Records
Examine dataset shape, column names, pandas data types, and preview the first 10 rows.


In [2]:
# Display dimensions and column names
print("=== DATASET DIMENSIONS ===")
print(f"Total Rows: {df_raw.shape[0]}")
print(f"Total Columns: {df_raw.shape[1]}")

print("\n=== COLUMN NAMES AND DATA TYPES ===")
schema_df = pd.DataFrame({
    'Column Name': df_raw.columns,
    'Data Type': df_raw.dtypes.astype(str).values,
    'Non-Null Count': df_raw.notnull().sum().values,
    'Null Count': df_raw.isnull().sum().values
})
print(schema_df.to_string(index=False))


=== DATASET DIMENSIONS ===
Total Rows: 2260
Total Columns: 13

=== COLUMN NAMES AND DATA TYPES ===
    Column Name Data Type  Non-Null Count  Null Count
        post_id       str            2260           0
      timestamp       str            2260           0
       platform       str            2260           0
           user       str            2260           0
followers_count       str            2170          90
       verified       str            2216          44
           text       str            2260           0
       hashtags       str            1472         788
        country       str            2134         126
          likes     int64            2260           0
retweets_shares     int64            2260           0
       comments   float64            2214          46
      sentiment       str            2227          33


In [3]:
# Display the first 10 rows of the dataset
print("=== FIRST 10 ROWS ===")
print(df_raw.head(10).to_string())


=== FIRST 10 ROWS ===
   post_id            timestamp   platform         user followers_count verified                                                                 text                   hashtags country  likes  retweets_shares  comments sentiment
0  P100040  2023-02-10 13:01:38     TikTok   quinn_5984           668.0       No                Reading more about service this week, lots to unpack.                    #Travel     USA     30                8       4.0   Neutral
1  P101934  2024-04-26 19:26:56   Facebook  taylor_6536            54.0       No      This event launch exceeded all my expectations, fantastic work.                        NaN  Canada     19                1       1.0  Positive
2  P100375  2023-06-29 17:51:22  Instagram    ravi_5073           559.0       No         Worst brand experience I've had in a long time, frustrating.                       #Art      UK     27                8       1.0  Negative
3  P101166  2023-02-25 03:21:56  Instagram   avery_3025       

## 3. Missing Value Audit
Quantify missing values across all columns, computing exact missing counts and percentages.


In [4]:
missing_counts = df_raw.isnull().sum()
missing_pct = (missing_counts / len(df_raw)) * 100

missing_audit = pd.DataFrame({
    'Total Records': len(df_raw),
    'Missing Count': missing_counts,
    'Missing Percentage (%)': missing_pct.round(2)
}).sort_values(by='Missing Count', ascending=False)

print("=== MISSING VALUE AUDIT SUMMARY ===")
print(missing_audit[missing_audit['Missing Count'] > 0].to_string())

total_missing_cells = df_raw.isnull().sum().sum()
total_cells = df_raw.size
print(f"\nTotal Missing Cells: {total_missing_cells:,} out of {total_cells:,} ({total_missing_cells/total_cells*100:.2f}%)")


=== MISSING VALUE AUDIT SUMMARY ===
                 Total Records  Missing Count  Missing Percentage (%)
hashtags                  2260            788                   34.87
country                   2260            126                    5.58
followers_count           2260             90                    3.98
comments                  2260             46                    2.04
verified                  2260             44                    1.95
sentiment                 2260             33                    1.46

Total Missing Cells: 1,127 out of 29,380 (3.84%)


## 4. Duplicate Records Audit
Analyze both full-row identical duplicates and semantic duplicates sharing the same `post_id`.


In [5]:
# 1. Exact Full-Row Duplicates
exact_dupes_count = df_raw.duplicated().sum()
print(f"Exact full-row duplicate records: {exact_dupes_count}")

# 2. Duplicate Post IDs
post_id_dupes_count = df_raw.duplicated(subset=['post_id']).sum()
print(f"Duplicate post_id count: {post_id_dupes_count}")
print(f"Total rows involved in duplicate post_id groups: {len(df_raw[df_raw.duplicated(subset=['post_id'], keep=False)])}")

# 3. Discrepancy analysis on differing post_id duplicates
all_dup_post_ids = df_raw[df_raw.duplicated(subset=['post_id'], keep=False)]
differing_mask = ~df_raw.duplicated(keep=False)
differing_dupes = all_dup_post_ids[differing_mask.loc[all_dup_post_ids.index]]
print(f"Rows with same post_id but differing column values: {len(differing_dupes)}")

# Sample differing pairs
differing_pids = differing_dupes['post_id'].unique()
print(f"Distinct post_ids with differing values: {len(differing_pids)}")
print("\nSample differing duplicate post_ids (showing format/casing discrepancy):")
print(differing_dupes[['post_id', 'platform', 'followers_count', 'country', 'likes', 'comments']].sort_values('post_id').head(10).to_string())


Exact full-row duplicate records: 44
Duplicate post_id count: 60
Total rows involved in duplicate post_id groups: 120
Rows with same post_id but differing column values: 32
Distinct post_ids with differing values: 16

Sample differing duplicate post_ids (showing format/casing discrepancy):
      post_id   platform followers_count country  likes  comments
399   P100007    TWITTER           764.0     NaN     32       1.0
1777  P100007    Twitter           764.0     NaN     32       1.0
180   P100054    Twitter         20454.0     NaN     47       NaN
323   P100054    Twitter          20,454     NaN     47       NaN
559   P100060    Twitter           310.0     NaN     41       2.0
1208  P100060    Twitter             310     NaN     41       2.0
1553  P100430  Instagram           508.0     NaN     19       2.0
1299  P100430  Instagram             508     NaN     19       2.0
2061  P100488    Twitter           1,524     NaN     26       2.0
1907  P100488    Twitter          1524.0     NaN 

## 5. Categorical Consistency Audit
Examine platform names, country designations, verification flags, and sentiment classifications for casing, whitespace padding, or invalid classes.


In [6]:
print("=== 1. PLATFORM COLUMN AUDIT ===")
print("Unique raw values:", repr(df_raw['platform'].unique().tolist()))
print("\nValue counts:")
print(df_raw['platform'].value_counts())

print("\n=== 2. COUNTRY COLUMN AUDIT ===")
print(f"Total unique raw entries (including uncleaned strings): {df_raw['country'].nunique(dropna=False)}")
print("\nRaw value counts:")
print(df_raw['country'].value_counts(dropna=False))

print("\n=== 3. VERIFIED COLUMN AUDIT ===")
print("Unique values:", repr(df_raw['verified'].unique().tolist()))
print(df_raw['verified'].value_counts(dropna=False))

print("\n=== 4. SENTIMENT COLUMN AUDIT ===")
print("Unique values:", repr(df_raw['sentiment'].unique().tolist()))
print(df_raw['sentiment'].value_counts(dropna=False))


=== 1. PLATFORM COLUMN AUDIT ===
Unique raw values: ['TikTok', 'Facebook', 'Instagram', 'LinkedIn', 'Twitter', 'TWITTER', 'INSTAGRAM', 'FACEBOOK', 'LINKEDIN', 'TIKTOK']

Value counts:
platform
Twitter      691
Instagram    561
TikTok       357
Facebook     352
LinkedIn     118
TWITTER       54
INSTAGRAM     53
TIKTOK        41
FACEBOOK      27
LINKEDIN       6
Name: count, dtype: int64

=== 2. COUNTRY COLUMN AUDIT ===
Total unique raw entries (including uncleaned strings): 25

Raw value counts:
country
USA               433
India             367
UK                202
Canada            164
Australia         156
Germany           143
NaN               126
UAE               110
France            104
Brazil            101
Japan             101
South Africa       78
Mexico             49
usa                33
india              22
uk                 15
canada             12
australia           8
uae                 7
brazil              6
germany             6
japan               5
france  

## 6. Numerical & Engagement Integrity Audit
Inspect engagement metrics (`likes`, `retweets_shares`, `comments`, and `followers_count`) for negative values, improper data types, and extreme outliers.


In [7]:
print("=== NUMERICAL DESCRIPTIVE STATISTICS ===")
print(df_raw[['likes', 'retweets_shares', 'comments']].describe().to_string())

# 1. Negative Value Check
print("\n=== NEGATIVE VALUES CHECK ===")
for col in ['likes', 'retweets_shares', 'comments']:
    neg_mask = df_raw[col] < 0
    print(f"Column '{col}' negative count: {neg_mask.sum()}")
    if neg_mask.sum() > 0:
        print(f"Displaying negative records in '{col}':")
        print(df_raw[neg_mask][['post_id', 'platform', 'country', col, 'retweets_shares', 'comments']].to_string())

# 2. followers_count Data Type Anomaly Check
print("\n=== FOLLOWERS_COUNT FORMATTING CHECK ===")
print(f"Data type in DataFrame: {df_raw['followers_count'].dtype}")
comma_followers = df_raw['followers_count'].dropna().apply(lambda x: ',' in str(x))
print(f"Records with comma formatting (e.g., '20,454'): {comma_followers.sum()}")
print("Sample comma-formatted followers values:", df_raw['followers_count'].dropna()[comma_followers].head(5).tolist())

# 3. Outlier Analysis using Interquartile Range (IQR)
print("\n=== ENGAGEMENT OUTLIERS (IQR METHOD) ===")
for metric in ['likes', 'retweets_shares', 'comments']:
    s = df_raw[metric].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    outliers = s[s > upper_fence]
    print(f"{metric:15s} | Q1: {q1:5.1f} | Median: {s.median():5.1f} | Q3: {q3:5.1f} | Upper Fence: {upper_fence:5.1f} | Outliers: {len(outliers):3d} ({len(outliers)/len(s)*100:4.1f}%) | Max: {s.max()}")


=== NUMERICAL DESCRIPTIVE STATISTICS ===
            likes  retweets_shares     comments
count  2260.00000      2260.000000  2214.000000
mean     35.94646         6.617699     3.448509
std      16.56231         4.721631     2.745874
min      -1.00000         0.000000     0.000000
25%      25.00000         3.000000     1.000000
50%      33.00000         6.000000     3.000000
75%      44.00000         9.000000     5.000000
max     126.00000        37.000000    20.000000

=== NEGATIVE VALUES CHECK ===
Column 'likes' negative count: 8
Displaying negative records in 'likes':
      post_id   platform       country  likes  retweets_shares  comments
157   P101989  Instagram       Germany     -1                4       3.0
196   P100866     TikTok         India     -1                4       8.0
1205  P100777   Facebook         India     -1                5       2.0
1495  P100121    Twitter     Australia     -1                8       0.0
1566  P100938    Twitter  South Africa     -1             

## 7. Temporal Integrity Audit
Verify date format parsing, identify time horizon, and ensure no invalid dates, future dates, or parsing failures exist.


In [8]:
# Timestamp parsing and date range check
parsed_dates = pd.to_datetime(df_raw['timestamp'], errors='coerce')
nat_count = parsed_dates.isna().sum()

print("=== TEMPORAL AUDIT ===")
print(f"Total timestamp entries: {len(df_raw)}")
print(f"Unparseable / NaT timestamps: {nat_count}")
print(f"Earliest timestamp in dataset: {parsed_dates.min()}")
print(f"Latest timestamp in dataset:   {parsed_dates.max()}")
print(f"Temporal span: {(parsed_dates.max() - parsed_dates.min()).days} days (2 full calendar years: 2023-2024)")

# Posts by year and month
posts_by_year = parsed_dates.dt.year.value_counts().sort_index()
print("\nPosts per calendar year:")
print(posts_by_year.to_string())


=== TEMPORAL AUDIT ===
Total timestamp entries: 2260
Unparseable / NaT timestamps: 0
Earliest timestamp in dataset: 2023-01-01 01:08:09
Latest timestamp in dataset:   2024-12-31 21:11:13
Temporal span: 730 days (2 full calendar years: 2023-2024)

Posts per calendar year:
timestamp
2023    1119
2024    1141


## 8. College DV Project Requirements Verification
Verification of specific data attributes required for the Social Media Engagement Analysis project.


In [9]:
project_requirements = {
    "1. Social media platform names": ("PRESENT", "Column 'platform' (5 platforms, casing normalization required)"),
    "2. Country": ("PRESENT", "Column 'country' (12 countries, whitespace & casing issues, 126 nulls)"),
    "3. State or region": ("MISSING", "Not available in dataset schema. Geographic analysis limited to country level."),
    "4. Post date or timestamp": ("PRESENT", "Column 'timestamp' (2023-01-01 to 2024-12-31, 0 nulls, 100% valid)"),
    "5. Likes": ("PRESENT", "Column 'likes' (Integer, 8 negative sentinel values of -1)"),
    "6. Comments": ("PRESENT", "Column 'comments' (Stored as float64, 46 nulls)"),
    "7. Shares / Retweets": ("PRESENT", "Column 'retweets_shares' (Integer, min 0, max 37, 0 nulls)"),
    "8. Views or impressions": ("MISSING", "Not available in dataset schema. Cannot calculate impression-based engagement rate."),
    "9. Content type": ("MISSING", "Not available in dataset schema. Cannot categorize by video/photo/carousel.")
}

req_df = pd.DataFrame([
    {"Requirement": k, "Status": v[0], "Details & Audit Findings": v[1]}
    for k, v in project_requirements.items()
])

print("=== SPECIFIC DV PROJECT REQUIREMENTS CHECKLIST ===")
print(req_df.to_string(index=False))


=== SPECIFIC DV PROJECT REQUIREMENTS CHECKLIST ===
                   Requirement  Status                                                            Details & Audit Findings
1. Social media platform names PRESENT                      Column 'platform' (5 platforms, casing normalization required)
                    2. Country PRESENT              Column 'country' (12 countries, whitespace & casing issues, 126 nulls)
            3. State or region MISSING      Not available in dataset schema. Geographic analysis limited to country level.
     4. Post date or timestamp PRESENT                  Column 'timestamp' (2023-01-01 to 2024-12-31, 0 nulls, 100% valid)
                      5. Likes PRESENT                          Column 'likes' (Integer, 8 negative sentinel values of -1)
                   6. Comments PRESENT                                     Column 'comments' (Stored as float64, 46 nulls)
          7. Shares / Retweets PRESENT                          Column 'retweets_shares'

## 9. Key Findings & Proposed Cleaning Treatments
Summary of identified data quality issues and proposed treatments for stakeholder approval before proceeding to data cleaning or visualization.

| Issue # | Column | Defect Description | Impact / Prevalence | Proposed Treatment (Pending User Approval) |
|---|---|---|---|---|
| **1** | `post_id` / Full rows | Duplicate records | 44 full duplicates; 16 semantic duplicates with platform case or followers formatting variations (Total: 60 redundant records) | Deduplicate by `post_id`, retaining the standardized canonical record. |
| **2** | `platform` | Inconsistent uppercase vs title case | 181 records have ALL-CAPS names (`TWITTER`, `INSTAGRAM`, `TIKTOK`, `FACEBOOK`, `LINKEDIN`) | Normalize with `.str.strip().str.title()`. |
| **3** | `country` | Trailing spaces and lowercase entries | 122 records have trailing spaces and lowercase (e.g. `'usa  '`, `'india  '`); 126 records are `NaN` (5.58%) | Strip whitespace, standardize title/uppercase formatting. Impute missing countries as `'Unknown'` or handle via null-safe category. |
| **4** | `likes` | Negative engagement count | 8 records contain `-1` (data recording error / sentinel) | Replace `-1` with `0` or median platform likes, or treat as missing (`NaN`). |
| **5** | `followers_count` | String object dtype due to commas and decimals | 67 values contain comma separators (`'20,454'`); 90 values are `NaN` (3.98%) | Strip commas, convert to numeric (`Int64` / `float64`), impute missing with median per platform/verification tier. |
| **6** | `comments` | Missing values and float64 type | 46 missing values (`NaN`, 2.04%) | Impute missing comments (e.g. 0 or platform median) and cast to integer. |
| **7** | `verified` | Missing values | 44 missing values (`NaN`, 1.95%) | Impute with mode (`'No'`) or assign explicit `'Unknown'` flag. |
| **8** | `sentiment` | Missing values | 33 missing values (`NaN`, 1.46%) | Retain as `'Unknown'` / unclassified or infer from text if requested. |
| **9** | Schema limitations | Absence of state/region, views, and content type | Limits geographic granularity to country-level; engagement rate must be follower-based or interaction-sum based. | Design visualizations tailored to available dimensions without hallucinating synthetic fields. |

> **Audit Status**: Inspection complete. Dataset has not been altered. Awaiting approval on proposed treatments before cleaning.
